In [ ]:
import sys
from pathlib import Path

import pandas as pd
import logging

%load_ext autoreload
%autoreload 2

# Repo layout: experimental/ sits next to this notebook's utils module.
HERE = Path.cwd()
if (HERE / 'wiki_parser_utils.py').exists():
    sys.path.insert(0, str(HERE))
elif (HERE / 'experimental' / 'wiki_parser_utils.py').exists():
    sys.path.insert(0, str(HERE / 'experimental'))

import wiki_parser_utils as wpu


In [ ]:
TARGET_URLS_CONFIG = {
    "https://en.wikipedia.org/wiki/List_of_20th-century_classical_composers": {"class": "wikitable sortable"}
}

# Optional: configure logging here if not handled adequately by wpu
# for handler in logging.root.handlers[:]:
#     logging.root.removeHandler(handler)
# logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def main_notebook_workflow():
    all_dfs = []
    individual_dfs_info = {}

    for url, table_selectors in TARGET_URLS_CONFIG.items():
        logging.info(f"Processing URL: {url}")
        soup = wpu.get_wiki_soup(url)

        if soup is not None:
            raw_data = wpu.parse_table(soup, table_selectors)
            if raw_data and raw_data:
                df = pd.DataFrame(raw_data)
                df['Source URL'] = url
                all_dfs.append(df)
                logging.info(f"Successfully extracted {len(df)} rows from {url}.")
                individual_dfs_info[url] = {
                    "columns": df.columns.tolist(),
                    "head": df.head().to_string()
                }
            else:
                logging.warning(f"No data extracted from the table in {url}.")
        else:
            logging.error(f"Failed to fetch or parse the webpage: {url}")

    logging.info("\n--- Individual DataFrame Inspection ---")
    for url, info in individual_dfs_info.items():
        logging.info(f"--- Info for: {url} ---")
        logging.info(f"Columns: {info['columns']}")
        logging.info(f"Head:\n{info['head']}")
        logging.info("--- End Info ---")

    if not all_dfs:
        logging.error("No data was extracted from any URL. Returning empty DataFrame.")
        return pd.DataFrame() # Return an empty DataFrame

    try:
        combined_df = pd.concat(all_dfs, ignore_index=True, sort=False)
    except Exception as e:
        logging.error(f"Error during pd.concat: {e}")
        return pd.DataFrame() # Return an empty DataFrame

    logging.info(f"Successfully created combined_df with {len(combined_df)} rows.")
    logging.info(f"Combined DataFrame columns: {combined_df.columns.tolist()}")

        # --- DEDUPLICATION STEP ---
    if 'URL' in combined_df.columns:
        logging.info(f"Number of rows before deduplication based on URL: {len(combined_df)}")
        # Sort by URL and then by the number of non-NaN values in birth/death years to prioritize more complete rows.
        # Create a temporary count of non-nulls for sorting.
        # Ensure 'Year of birth' and 'Year of death' exist from parsing before trying to count NaNs.
        # The actual year parsing happens in clean_data, so here we might not have them as numeric yet.
        # For simplicity, let's just keep the first occurrence after sorting by Source URL (optional, if you prefer one source)
        # A more robust way is to sort by completeness AFTER cleaning, then deduplicate.
        # But for now, let's deduplicate on URL, keeping the first instance.
        # This assumes the first instance encountered (e.g., from 20th C list) is good enough or will be cleaned.

        # Count NaN values in crucial columns for each row
        # We need to ensure these columns exist before trying to count NaNs on them.
        # This is better done *after* clean_data, but let's try a simple dedupe first.
        
        num_duplicates_on_url = combined_df.duplicated(subset=['URL'], keep=False).sum()
        logging.info(f"Number of rows involved in URL duplicates (will be > count of unique duplicated URLs): {num_duplicates_on_url}")
        
        # To keep the row with more information (fewer NaNs) when duplicates occur on URL:
        # 1. Create a 'completeness_score' (e.g., count of non-nulls in important fields)
        # 2. Sort by URL and then by this score (descending)
        # 3. Drop duplicates on URL, keeping 'first' (which will be the most complete)

        # For simplicity for now, just keep=first.
        # If you find that the "first" entry is less complete than a "last" entry for the same URL,
        # you can change keep='last' or implement a more sophisticated merge.
        combined_df.sort_values(by=['URL', 'Year of birth'], na_position='last', inplace=True) # Prioritize rows with birth year
        combined_df.drop_duplicates(subset=['URL'], keep='first', inplace=True)
        logging.info(f"Number of rows AFTER deduplication based on URL: {len(combined_df)}")
    else:
        logging.warning("'URL' column not found, skipping deduplication. Duplicates may exist.")
    # --- END DEDUPLICATION ---
    
    df_cleaned = wpu.clean_data_from_parsed_years(combined_df.copy())

    print("\n--- Cleaned DataFrame with Date Info (First 5 rows) ---")
    # ... (rest of the print and display statements for df_cleaned) ...
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    
    cols_to_show = ['Name', 'Year of birth', 'Year of death', 'Potential EU PD Year', 'Nationality', 'URL', 'Source URL']
    cols_to_show_existing = [col for col in cols_to_show if col in df_cleaned.columns]
    if not df_cleaned.empty:
        display(df_cleaned[cols_to_show_existing].head())
        print(f"\nTotal composers processed: {len(df_cleaned)}")
        
        valid_birth_years = df_cleaned['Year of birth'].notna().sum() if 'Year of birth' in df_cleaned else 0
        valid_death_years = df_cleaned['Year of death'].notna().sum() if 'Year of death' in df_cleaned else 0
        logging.info(f"Number of valid birth years in cleaned_df: {valid_birth_years}")
        logging.info(f"Number of valid death years in cleaned_df: {valid_death_years}")

        if 'Potential EU PD Year' in df_cleaned and valid_death_years > 0:
            current_year = pd.Timestamp.now().year
            df_cleaned['Potential EU PD Year'] = pd.to_numeric(df_cleaned['Potential EU PD Year'], errors='coerce')
            
            potentially_pd_composers = df_cleaned[
                df_cleaned['Potential EU PD Year'].notna() &
                (df_cleaned['Potential EU PD Year'] <= current_year)
            ]
            print(f"\nComposers potentially in EU Public Domain by {current_year}: {len(potentially_pd_composers)}")
            if not potentially_pd_composers.empty:
                display(potentially_pd_composers[cols_to_show_existing].head())

            died_20th_century = df_cleaned[
                df_cleaned['Year of death'].notna() &
                (df_cleaned['Year of death'] >= 1901) & (df_cleaned['Year of death'] <= 2000)
            ]
            print(f"\nComposers who died in the 20th century (1901-2000): {len(died_20th_century)}")
            if not died_20th_century.empty:
                display(died_20th_century[cols_to_show_existing].head())
        else:
            logging.warning("Skipping PD calculations as 'Potential EU PD Year' or valid death years are missing.")
    else:
        logging.warning("Cleaned DataFrame is empty.")

    return df_cleaned

In [ ]:
composers_df = main_notebook_workflow()

In [ ]:
if composers_df is not None and not composers_df.empty:
    print("\n--- Accessing the DataFrame for further use: ---")
    display(composers_df.info())
    display(composers_df.head())
    
    # Example: Save to CSV
    # composers_df.to_csv("all_composers_cleaned.csv", index=False)
    # logging.info("DataFrame saved to all_composers_cleaned.csv")

    # Example: How many unique nationalities?
    # print("\nUnique Nationalities:")
    # print(composers_df['Nationality'].value_counts().head(10))
else:
    print("composers_df is empty or None, cannot perform further operations.")

In [ ]:
# Cell 5: Fetch Pageviews (Modified for Subset with Configurable Death Year Range)

from tqdm import tqdm
import pandas as pd
import numpy as np
import logging # Ensure logging is imported if not already
# Assuming wpu (wiki_parser_utils) is imported in a previous cell
# and wpu.get_pageviews_for_url is defined there.

# --- Configuration for Pageviews & Subset ---
PAGEVIEWS_START_DATE = "20240101"
PAGEVIEWS_END_DATE = "20241231"

# Define the death year range for selecting the subset for pageview fetching
# Set to None if you don't want to use a particular bound.
# Example 1: Died before 1960 (exclusive of 1960)
DEATH_YEAR_MIN = None  # No minimum death year for this example
DEATH_YEAR_MAX = 1960  # Composers must have died *before* this year

# Example 2: Died between 1920 and 1950 (inclusive)
# DEATH_YEAR_MIN = 1920
# DEATH_YEAR_MAX = 1951 # To make it inclusive of 1950 (died < 1951)

# Example 3: Died in or after 1950
# DEATH_YEAR_MIN = 1950
# DEATH_YEAR_MAX = None # No maximum death year

# Example 4: Fetch for all (effectively no death year filter for pageviews)
# DEATH_YEAR_MIN = None
# DEATH_YEAR_MAX = None

# --- End Configuration ---


# Check if composers_df exists and is valid
if 'composers_df' not in locals() or not isinstance(composers_df, pd.DataFrame) or composers_df.empty:
    logging.error("composers_df is not defined, empty, or not a DataFrame. Run previous cells to create it.")
else:
    if 'URL' not in composers_df.columns:
        logging.error("The 'URL' column is missing from composers_df. Cannot fetch pageviews.")
    elif 'Year of death' not in composers_df.columns:
        logging.error("The 'Year of death' column is missing from composers_df. Cannot create subset for pageviews.")
    else:
        # --- Create the Subset for Pageview Fetching ---
        # Ensure 'Year of death' is numeric for filtering.
        # This should have been done when composers_df was created/cleaned.
        # We'll coerce again here just to be safe for the filtering logic.
        year_of_death_numeric = pd.to_numeric(composers_df['Year of death'], errors='coerce')
        
        # Start with a mask that includes all rows (all True)
        subset_mask = pd.Series(True, index=composers_df.index)
        
        filter_description_parts = []

        if DEATH_YEAR_MIN is not None:
            subset_mask &= (year_of_death_numeric.notna()) & (year_of_death_numeric >= DEATH_YEAR_MIN)
            filter_description_parts.append(f"died on/after {DEATH_YEAR_MIN}")
        
        if DEATH_YEAR_MAX is not None:
            subset_mask &= (year_of_death_numeric.notna()) & (year_of_death_numeric < DEATH_YEAR_MAX)
            filter_description_parts.append(f"died before {DEATH_YEAR_MAX}")

        if not filter_description_parts: # No year filters applied
            filter_description = "all composers (no death year filter for pageviews)"
        else:
            filter_description = " and ".join(filter_description_parts)
        
        composers_subset_for_pageviews = composers_df[subset_mask]

        logging.info(f"Full composers_df has {len(composers_df)} entries.")
        logging.info(f"Subset for pageview fetching ({filter_description}): {len(composers_subset_for_pageviews)} composers.")

        if not composers_subset_for_pageviews.empty:
            tqdm.pandas(desc=f"Fetching Pageviews ({filter_description})")
            
            pageviews_series_for_subset = composers_subset_for_pageviews['URL'].progress_apply(
                wpu.get_pageviews_for_url,
                start_date=PAGEVIEWS_START_DATE,
                end_date=PAGEVIEWS_END_DATE
            )
            
            if 'Pageviews' not in composers_df.columns:
                composers_df['Pageviews'] = np.nan
            
            composers_df.loc[subset_mask, 'Pageviews'] = pageviews_series_for_subset
            logging.info(f"Pageviews fetched and updated for {pageviews_series_for_subset.notna().sum()} composers in the subset.")
        else:
            logging.info("Subset for pageview fetching is empty based on death year criteria. No pageviews fetched.")
            if 'Pageviews' not in composers_df.columns:
                 composers_df['Pageviews'] = np.nan

        # --- Proceed with sorting and displaying using the full composers_df ---
        composers_df_sorted_by_views = composers_df.sort_values(by='Pageviews', ascending=False, na_position='last')

        print(f"\n--- Top Composers (from subset: {filter_description}) with Pageviews ---")
        cols_to_show_pv = ['Name', 'Pageviews', 'Year of birth', 'Year of death', 'Nationality', 'URL']
        cols_to_show_pv_existing = [col for col in cols_to_show_pv if col in composers_df_sorted_by_views.columns]
        
        display(composers_df_sorted_by_views[composers_df_sorted_by_views['Pageviews'].notna()][cols_to_show_pv_existing].head(20))

        if not composers_subset_for_pageviews.empty:
            nan_in_subset_pageviews = pageviews_series_for_subset.isna().sum()
            if nan_in_subset_pageviews > 0:
                logging.warning(f"{nan_in_subset_pageviews} composers in the SUBSET had issues fetching pageviews (NaN result). Check logs for errors.")
        
        total_pageview_nan_count = composers_df['Pageviews'].isna().sum()
        logging.info(f"Total NaNs in 'Pageviews' column (including those not in subset or due to errors): {total_pageview_nan_count}")

In [ ]:
# Cell 6: Accessing and Analyzing the Subset with Pageviews

if 'composers_df' in locals() and isinstance(composers_df, pd.DataFrame) and 'Pageviews' in composers_df.columns:
    # Create a DataFrame containing only composers for whom pageviews were fetched (i.e., Pageviews is not NaN)
    composers_with_pageviews_df = composers_df[composers_df['Pageviews'].notna()].copy() # .copy() if you plan to modify it

    if not composers_with_pageviews_df.empty:
        print(f"Successfully accessed subset of {len(composers_with_pageviews_df)} composers with pageview data.")
        
        print("\n--- Head of the subset with pageviews ---")
        display(composers_with_pageviews_df[['Name', 'Year of death', 'Pageviews', 'URL']].head())

        # Now you can perform further analysis specifically on this subset:
        # For example, sort this subset by pageviews
        subset_sorted = composers_with_pageviews_df.sort_values(by='Pageviews', ascending=False)
        print("\n--- Top composers from the subset (sorted again for clarity) ---")
        display(subset_sorted[['Name', 'Year of death', 'Pageviews']].head(20))

        # Example: Get basic statistics for pageviews in this subset
        print("\n--- Pageview Statistics for the Subset ---")
        display(subset_sorted['Pageviews'].describe())
        
    else:
        print("No composers with pageview data found (Pageviews column is all NaN or subset was empty).")
else:
    print("composers_df or 'Pageviews' column not found. Please run previous cells.")

In [ ]:
display(composers_with_pageviews_df)